In [ ]:
from miniproto1 import *
import numpy as np
import random
import pandas as pd
import time
import matplotlib.pyplot as plt

# Experimentació: Explotació de l'Espai de Casos

La idea d'aquest apartat de l'experimentació serà intentar afirmar la següent hipòtesi:

- Si quan un cas és considerat redundant, no comprovem que la valoració sigui millor o pitjor, llavors si generem blocs de casos aleatoris indefinidament, hi haurà un punt on el nombre de casos afegits per bloc serà igual a zero. 

Això serà degut a què tot l'espai de casos ha estat cobert i per tant considera tots els casos com a redundants.

Posteriorment veurem què passa quan sí comprovem si hi ha una millora o no de la valoració. Això farà que s'intercanvïin els casos i, per tant, millori la qualitat de les recomanacions. Alhora trigarà més en arribar el punt on el nombre de casos afegits per bloc sigui zero. A més a més, aquell punt serà en el què el nostre CBR sigui tot el _perfecte_ que les nostres dades ho permeten. 

In [ ]:
def block(rs:RecommenderSystem, blocksize:int=100):
    
    redundants = 0
    for _ in range(blocksize):
        newp = eval(generate_new_problem())
        newcas = Cas(id=(rs.indexacio.ultim_id),descripcio=rs.scaler.transform([newp.array])[0])    
        
        most_similar_cases = rs.retrieve(newp)
        distancies, casos, paths = zip(*most_similar_cases)
        cas_dists = list(zip(distancies, casos))
        dicc_most_sim = rs.reuse(cas_dists, newcas)
        rs.revise(newcas, casos, dicc_most_sim, False) 
        redundancia = rs.retain(newcas, cas_dists, paths[0], 2)
        if redundancia == 1:
            redundants += 1

    return redundants/blocksize

In [ ]:
df_problemes = pd.read_csv('./data/problems4000_not_0_0_0.csv')
rs = createRS(
    from_model= False, 
    casebase=df_problemes, 
    maxcasesxleaf=15
)

num_blocks = 100
rates = []

for b in range(num_blocks):
    rate = block(rs)
    rates.append(rate)
    print(f'Rati en el block {b} = {rate}')